# 06 · Immune cell infiltration — macrophage sub-cluster analysis
**Input:** `data/processed/adata_annotated.h5ad`

This notebook goes beyond the original Wang et al. (2025) paper by performing
a detailed sub-cluster analysis of tumor-associated macrophages (TAMs).
It scores each macrophage for M1 (pro-inflammatory), M2 (anti-inflammatory),
Kupffer cell, and TAM signatures, then compares normal vs tumor macrophage
composition and gene expression.

*This is original analytical work not present in the paper.*


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
adata = sc.read("../data/processed/adata_annotated.h5ad")
print(adata)

## Subset macrophages

In [ ]:
adata_mac = adata[adata.obs["manual_celltype"] == "Macrophage"].copy()
print(f"Macrophages: {adata_mac.n_obs} cells")
print(adata_mac.obs["sample"].value_counts())

## Re-cluster macrophages

In [ ]:
sc.pp.neighbors(adata_mac, n_neighbors=10, n_pcs=10)
sc.tl.umap(adata_mac)
sc.tl.leiden(adata_mac, key_added="mac_cluster", resolution=0.4, flavor="igraph")
sc.pl.umap(adata_mac, color=["mac_cluster","sample"], wspace=0.4)

## Module scoring: M1 / M2 / Kupffer / TAM

In [ ]:
MAC_SIGNATURES = {
    "M1_score"     : ["TNF","IL1B","CXCL9","CD86","STAT1","NOS2","CXCL10"],
    "M2_score"     : ["CD163","MRC1","IL10","VEGFA","APOE","ARG1","TGFB1"],
    "Kupffer_score": ["VSIG4","TIMD4","C1QA","C1QB","MARCO","CLEC4F","FOLR2"],
    "TAM_score"    : ["SPP1","TREM2","GPNMB","CD9","MMP9","VCAN","SLC40A1"],
}

for sig_name, gene_list in MAC_SIGNATURES.items():
    present = [g for g in gene_list if g in adata_mac.var_names]
    if present:
        sc.tl.score_genes(adata_mac, gene_list=present, score_name=sig_name)

sc.pl.umap(adata_mac,
           color=["M1_score","M2_score","Kupffer_score","TAM_score","sample"],
           ncols=3, wspace=0.4, frameon=False, cmap="RdYlBu_r",
           title=["M1 (pro-inflam.)","M2 (anti-inflam.)",
                  "Kupffer cells","TAMs","Sample"])

## Differential gene expression: normal vs tumor macrophages

In [ ]:
sc.tl.rank_genes_groups(adata_mac, groupby="sample",
                        method="wilcoxon", key_added="mac_de")

mac_de = sc.get.rank_genes_groups_df(adata_mac, group="tumor (HCC2)", key="mac_de")
mac_de_sig = mac_de[(mac_de["pvals_adj"]<0.05) & (mac_de["logfoldchanges"].abs()>1)]

out = Path("../results/tables/macrophage_tumor_vs_normal_DE.csv")
mac_de_sig.to_csv(out, index=False)
print(f"Macrophage DEGs: {len(mac_de_sig)}")
print("\nTop upregulated in tumour macrophages:")
print(mac_de_sig.sort_values("logfoldchanges",ascending=False).head(10)[["names","logfoldchanges","pvals_adj"]])

## Visualise TAM vs Kupffer signatures

In [ ]:
vis_genes = {
    "Kupffer / Resident": ["VSIG4","TIMD4","C1QA","C1QB","MARCO","CLEC4F"],
    "TAM / Tumour"       : ["SPP1","TREM2","GPNMB","CD9","MMP9","VCAN"],
    "M1 Activation"      : ["TNF","IL1B","CXCL9","CD86","STAT1"],
    "M2 / Immunosuppressive": ["CD163","MRC1","IL10","VEGFA","APOE","ARG1"],
    "Shared Myeloid"     : ["CD68","LYZ","CSF1R","S100A9","S100A8"],
}
flat_genes = [g for gs in vis_genes.values() for g in gs if g in adata_mac.var_names]
sc.pl.dotplot(adata_mac, flat_genes, groupby=["mac_cluster","sample"],
              standard_scale="var", figsize=(20,6),
              title="Macrophage signature genes per sub-cluster and sample")

## Macrophage composition: normal vs tumour

In [ ]:
comp = (
    adata_mac.obs.groupby(["sample","mac_cluster"])
    .size().unstack(fill_value=0)
    .div(adata_mac.obs["sample"].value_counts(), axis=0) * 100
)

fig, ax = plt.subplots(figsize=(10,4))
comp.T.plot(kind="bar", ax=ax, colormap="Set2", edgecolor="white", width=0.7)
ax.set_ylabel("% of macrophages")
ax.set_title("Macrophage sub-cluster composition: Normal (HCC1) vs Tumour (HCC2)")
ax.set_xlabel("Macrophage sub-cluster")
plt.xticks(rotation=0)
plt.legend(title="Sample", bbox_to_anchor=(1.05,1))
plt.tight_layout()
fig.savefig("../results/figures/macrophage_composition.png", dpi=200, bbox_inches="tight")
plt.show()

# Score summary
score_cols = [c for c in adata_mac.obs.columns if c.endswith("_score")]
if score_cols:
    print("\nMean module scores per sample:")
    print(adata_mac.obs.groupby("sample")[score_cols].mean().round(4))